# Spoon vs Fork — ML experiments notebook
A tiny CNN that classifies my own photos (30 spoons + 30 forks).  
The point is **not** to get a perfect classifier — it's to change one setting at a time and watch how training behaves.

**How to use this notebook:** run the cells top to bottom (Shift+Enter). Steps 1–4 set everything up. Steps 5–9 are the experiments. Step 10 saves all the plots so I can put them in my GitHub repo.

## Step 0 — Setup
This installs HEIC support (for iPhone photos) and imports everything. If your photos are already .jpg/.png you don't strictly need the HEIC line, but it's harmless.

In [ ]:
# iPhone photos are often .HEIC — this lets PIL read them. Skip-safe if not needed.
!pip -q install pillow-heif
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    print("HEIC support enabled")
except Exception as e:
    print("HEIC support not loaded (fine if your photos are jpg/png):", e)

import torch, torch.nn as nn, numpy as np, random, os, matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader, Subset

# Make results reproducible
SEED = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
os.makedirs("results/plots", exist_ok=True)
print("PyTorch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

## Step 1 — Load my dataset
Upload your photo zips using the folder icon on the left. This notebook accepts **either**:
- one `dataset.zip` that contains `fork/` and `spoon/` subfolders, **or**
- two separate zips whose names contain the words **fork** and **spoon** (e.g. a Google Photos download).

This cell figures out which you have, converts every photo to a clean PNG (so iPhone HEIC works too), and shows a few so I can confirm they're labelled correctly.

In [ ]:
import zipfile, shutil, glob
from PIL import Image

IMG = 64  # all images resized to 64x64
load_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor()])
IMG_EXT = (".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp", ".bmp", ".tif", ".tiff")

def build_class_folder(zip_paths, target):
    """Extract every image from the given zips into `target` as numbered PNGs."""
    os.makedirs(target, exist_ok=True)
    n = 0
    for zp in zip_paths:
        tmp = "/content/_tmp_extract"
        shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
        with zipfile.ZipFile(zp) as z: z.extractall(tmp)
        for root, _, files in os.walk(tmp):
            for fn in sorted(files):
                if fn.lower().endswith(IMG_EXT) and not fn.startswith("."):
                    try:
                        Image.open(os.path.join(root, fn)).convert("RGB").save(f"{target}/{n:04d}.png")
                        n += 1
                    except Exception as e:
                        print("  skipped", fn, "-", e)
        shutil.rmtree(tmp, ignore_errors=True)
    return n

shutil.rmtree("dataset", ignore_errors=True)
zips = glob.glob("/content/*.zip")
single = [z for z in zips if "dataset" in os.path.basename(z).lower()]

if single:                                   # case 1: one dataset.zip with fork/ and spoon/ inside
    with zipfile.ZipFile(single[0]) as z: z.extractall(".")
    shutil.rmtree("dataset/__MACOSX", ignore_errors=True)
    print("Loaded from", os.path.basename(single[0]))
else:                                        # case 2: separate fork / spoon zips
    fork_zips  = [z for z in zips if "fork"  in os.path.basename(z).lower()]
    spoon_zips = [z for z in zips if "spoon" in os.path.basename(z).lower()]
    nf = build_class_folder(fork_zips,  "dataset/fork")
    ns = build_class_folder(spoon_zips, "dataset/spoon")
    print(f"Built dataset from your zips:  {nf} fork  +  {ns} spoon")

full = ImageFolder("dataset", transform=load_tf)
print("Classes found:", full.classes, "| total images:", len(full))

import random as _r
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, i in zip(axes.flat, _r.sample(range(len(full)), 8)):
    img, label = full[i]
    ax.imshow(img.permute(1, 2, 0)); ax.set_title(full.classes[label]); ax.axis("off")
plt.tight_layout(); plt.show()

## Step 2 — Split into training and validation sets
**Training set** = the photos the model learns from.  
**Validation set** = photos it never trains on, used as a mock-exam to check if it actually learned vs just memorised.  
I use an 80/20 split.

In [ ]:
n_val = max(2, int(0.2 * len(full)))
n_train = len(full) - n_val
train_ds, val_ds = random_split(full, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))
print(f"Training images: {n_train}   Validation images: {n_val}")

## Step 3 — Define the small CNN
A small convolutional neural network: 2 convolution layers (which detect shapes/edges) then 2 fully-connected layers that make the final spoon-vs-fork decision.  
`dropout` and `use_bn` (batch-norm) are switches I'll flip in later experiments.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, dropout=0.0, use_bn=False):
        super().__init__()
        self.use_bn = use_bn
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1);  self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1); self.bn2 = nn.BatchNorm2d(32)
        self.pool  = nn.MaxPool2d(2)
        self.drop  = nn.Dropout(dropout)
        self.relu  = nn.ReLU()
        self.fc1   = nn.Linear(32 * 16 * 16, 64)
        self.fc2   = nn.Linear(64, 2)            # 2 outputs: fork or spoon
    def forward(self, x):
        x = self.conv1(x); x = self.bn1(x) if self.use_bn else x; x = self.pool(self.relu(x))
        x = self.conv2(x); x = self.bn2(x) if self.use_bn else x; x = self.pool(self.relu(x))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)
print("Model defined.")

## Step 4 — The training function + plotting helpers
`run_experiment(config)` trains a fresh model with the settings in `config` and returns the history (train/val loss and accuracy at every epoch). It also detects if training **diverged** (loss blew up to infinity).

In [ ]:
def run_experiment(config, train_subset=None):
    """Train a fresh model. config keys: optimizer, lr, batch_size, epochs, dropout, weight_decay, use_bn."""
    torch.manual_seed(SEED)
    tds = train_subset if train_subset is not None else train_ds
    model = SmallCNN(dropout=config.get("dropout", 0.0), use_bn=config.get("use_bn", False))
    tl = DataLoader(tds, batch_size=config["batch_size"], shuffle=True)
    vl = DataLoader(val_ds, batch_size=config["batch_size"])
    crit = nn.CrossEntropyLoss()
    wd = config.get("weight_decay", 0.0)
    if config["optimizer"] == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=config["lr"], weight_decay=wd)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=wd)

    h = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "diverged": False}
    for ep in range(config["epochs"]):
        model.train(); s = c = n = 0
        for xb, yb in tl:
            opt.zero_grad(); out = model(xb); loss = crit(out, yb)
            loss.backward(); opt.step()
            s += loss.item() * xb.size(0); c += (out.argmax(1) == yb).sum().item(); n += xb.size(0)
        model.eval(); vs = vc = vn = 0
        with torch.no_grad():
            for xb, yb in vl:
                out = model(xb); loss = crit(out, yb)
                vs += loss.item() * xb.size(0); vc += (out.argmax(1) == yb).sum().item(); vn += xb.size(0)
        trl, vll = s / n, vs / vn
        h["train_loss"].append(trl); h["val_loss"].append(vll)
        h["train_acc"].append(c / n); h["val_acc"].append(vc / vn)
        if not np.isfinite(trl) or trl > 1e4:        # loss exploded
            h["diverged"] = True
            print(f"   !! DIVERGED at epoch {ep+1} (train_loss={trl:.1e})")
            break
    tag = "DIVERGED" if h["diverged"] else f"train_acc={h['train_acc'][-1]:.2f}  val_acc={h['val_acc'][-1]:.2f}"
    print(f"   done: {tag}")
    return h

def plot_history(h, title, savepath):
    """Plot train/val loss and train/val accuracy side by side, and save the figure."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(h["train_loss"], label="train loss"); a1.plot(h["val_loss"], label="val loss")
    a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.set_title("Loss"); a1.legend(); a1.grid(alpha=.3)
    a2.plot(h["train_acc"], label="train acc"); a2.plot(h["val_acc"], label="val acc")
    a2.set_xlabel("epoch"); a2.set_ylabel("accuracy"); a2.set_title("Accuracy"); a2.legend(); a2.grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(savepath, dpi=110, bbox_inches="tight"); plt.show()
    print("saved ->", savepath)
print("Helpers ready.")

## Step 5 — Experiment 1: Baseline (sensible settings)
Adam optimizer, learning rate 0.001, batch size 8. This is my "known-good" run to compare everything else against. I expect loss to fall and accuracy to climb.

In [ ]:
baseline = {"optimizer": "adam", "lr": 0.001, "batch_size": 8, "epochs": 25, "dropout": 0.0}
print("Baseline:")
h_base = run_experiment(baseline)
plot_history(h_base, "Experiment 1 — Baseline (Adam, lr=0.001, batch=8)", "results/plots/01_baseline.png")

## Step 6 — Experiment 2: Learning-rate sweep (this contains my FAILED experiment)
Same model, three learning rates:
- **too low** (0.0001) — should crawl,
- **good** (0.001),
- **too high** (10.0 with SGD) — should **explode**.

Because the exploding loss is millions of times bigger than the others, the loss plot uses a **log scale** so all three are visible.

In [ ]:
lr_runs = {
    "too low (0.0001)":  {"optimizer": "adam", "lr": 0.0001, "batch_size": 8, "epochs": 25},
    "good (0.001)":      {"optimizer": "adam", "lr": 0.001,  "batch_size": 8, "epochs": 25},
    "too high (sgd 10.0)":{"optimizer": "sgd",  "lr": 10.0,    "batch_size": 8, "epochs": 25},
}
lr_hist = {}
for name, cfg in lr_runs.items():
    print(name); lr_hist[name] = run_experiment(cfg)

plt.figure(figsize=(7, 5))
for name, h in lr_hist.items():
    plt.plot(range(1, len(h["train_loss"]) + 1), h["train_loss"], marker="o", label=name)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("train loss (log scale)")
plt.title("Experiment 2 — Learning rate: too low vs good vs too high")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("results/plots/02_learning_rate.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved -> results/plots/02_learning_rate.png")

## Step 7 — Experiment 3: Overfitting vs Regularization
To *force* overfitting I deliberately train on only a small slice of the data (so the model can memorise it) for many epochs:
- **Run A (no regularization):** expect train accuracy → ~100% but a gap where validation is worse.
- **Run B (dropout 0.5 + weight decay):** the anti-memorising tricks should shrink that gap.

If both runs reach high validation accuracy, that's a real finding too — it means spoon-vs-fork was easy/clean enough that my model generalised well. Write down whatever actually happens.

In [ ]:
# Use a small training subset to encourage overfitting
small_n = min(16, n_train)
small_train = Subset(train_ds, list(range(small_n)))
print(f"Overfitting demo uses only {small_n} training images.\n")

overfit_cfg = {"optimizer": "adam", "lr": 0.001, "batch_size": 4, "epochs": 60, "dropout": 0.0, "weight_decay": 0.0}
reg_cfg     = {"optimizer": "adam", "lr": 0.001, "batch_size": 4, "epochs": 60, "dropout": 0.5, "weight_decay": 1e-3}

print("Run A — no regularization:"); h_over = run_experiment(overfit_cfg, train_subset=small_train)
print("Run B — dropout + weight decay:"); h_reg = run_experiment(reg_cfg, train_subset=small_train)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(h_over["train_loss"], label="train"); a1.plot(h_over["val_loss"], label="val")
a1.set_title("Run A: no regularization"); a1.set_xlabel("epoch"); a1.set_ylabel("loss"); a1.legend(); a1.grid(alpha=.3)
a2.plot(h_reg["train_loss"], label="train"); a2.plot(h_reg["val_loss"], label="val")
a2.set_title("Run B: dropout + weight decay"); a2.set_xlabel("epoch"); a2.set_ylabel("loss"); a2.legend(); a2.grid(alpha=.3)
fig.suptitle("Experiment 3 — Overfitting (A) vs Regularization (B)"); plt.tight_layout()
plt.savefig("results/plots/03_overfitting.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved -> results/plots/03_overfitting.png")

## Step 8 — Experiment 4: Batch size
How many photos the model looks at before each update.
- **batch = 2** — few photos per step → noisier, jumpier loss curve.
- **batch = 32** — many photos per step → smoother but slower-moving curve.

In [ ]:
bs_hist = {}
for bs in [2, 32]:
    print(f"batch size = {bs}")
    bs_hist[bs] = run_experiment({"optimizer": "adam", "lr": 0.001, "batch_size": bs, "epochs": 25})

plt.figure(figsize=(7, 5))
for bs, h in bs_hist.items():
    plt.plot(h["train_loss"], label=f"batch={bs}")
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.title("Experiment 4 — Batch size effect")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("results/plots/04_batch_size.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved -> results/plots/04_batch_size.png")

## Step 9 — Experiment 5: Optimizer (SGD vs Adam)
Same everything, two different "downhill strategies". Adam adapts its step sizes automatically; plain SGD doesn't. I expect Adam to reach low loss faster.

In [ ]:
opt_hist = {}
for name, cfg in {"SGD (lr=0.01)": {"optimizer": "sgd", "lr": 0.01, "batch_size": 8, "epochs": 25},
                  "Adam (lr=0.001)": {"optimizer": "adam", "lr": 0.001, "batch_size": 8, "epochs": 25}}.items():
    print(name); opt_hist[name] = run_experiment(cfg)

plt.figure(figsize=(7, 5))
for name, h in opt_hist.items():
    plt.plot(h["train_loss"], label=name)
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.title("Experiment 5 — SGD vs Adam")
plt.legend(); plt.grid(alpha=.3)
plt.savefig("results/plots/05_optimizer.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved -> results/plots/05_optimizer.png")

## Step 1 — Load my dataset
Upload your photo zips using the folder icon on the left. This notebook accepts **either**:
- one `dataset.zip` that contains `fork/` and `spoon/` subfolders, **or**
- two separate zips whose names contain the words **fork** and **spoon** (e.g. a Google Photos download).

This cell figures out which you have, converts every photo to a clean PNG (so iPhone HEIC works too), and shows a few so I can confirm they're labelled correctly.

In [ ]:
import csv
rows = [
    ("baseline",        baseline["lr"], baseline["batch_size"], "adam", h_base["train_acc"][-1], h_base["val_acc"][-1], h_base["diverged"]),
    ("lr_too_high",     10.0, 8, "sgd", lr_hist["too high (sgd 10.0)"]["train_acc"][-1], lr_hist["too high (sgd 10.0)"]["val_acc"][-1], lr_hist["too high (sgd 10.0)"]["diverged"]),
    ("overfit_no_reg",  0.001, 4, "adam", h_over["train_acc"][-1], h_over["val_acc"][-1], h_over["diverged"]),
    ("regularized",     0.001, 4, "adam", h_reg["train_acc"][-1], h_reg["val_acc"][-1], h_reg["diverged"]),
]
with open("results/summary.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["experiment", "lr", "batch", "optimizer", "final_train_acc", "final_val_acc", "diverged"])
    for r in rows: w.writerow([r[0], r[1], r[2], r[3], f"{r[4]:.2f}", f"{r[5]:.2f}", r[6]])
print("Wrote results/summary.csv")

# Zip results and download (Colab only)
import shutil
shutil.make_archive("results_for_github", "zip", "results")
try:
    from google.colab import files
    files.download("results_for_github.zip")
    print("Downloading results_for_github.zip — add these to your GitHub repo.")
except Exception as e:
    print("If not on Colab, grab results_for_github.zip from the file panel.", e)